# Gradients — companion notebook

Runnable companion for [`gradients.md`](./gradients.md).

1. Symbolic check of a gradient identity with SymPy.
2. Gradient field + level set visualization showing $\nabla f \perp \{f = c\}$.
3. Hand-rolled gradient descent on a 2D quadratic.
4. Numerical-vs-analytic gradient check (the standard backprop sanity test).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

rng = np.random.default_rng(0)

## 1. Symbolic check

Verify $\nabla_\mathbf{x} (\mathbf{x}^\top A \mathbf{x}) = (A + A^\top) \mathbf{x}$.

In [ ]:
x = sp.symbols('x1 x2 x3')
X = sp.Matrix(x)
A = sp.Matrix([[1, 1, 0],
               [2, 3, 1],
               [0, 1, 4]])
f = (X.T * A * X)[0, 0]
grad_symbolic = sp.Matrix([sp.diff(f, xi) for xi in x])
grad_identity = (A + A.T) * X
print('symbolic grad - (A + A^T) x =', sp.simplify(grad_symbolic - grad_identity).T)

## 2. Gradient field and level sets

We use $f(x, y) = x^2 + 3y^2$ (a tilted bowl). At every point the gradient arrow is perpendicular to the local level curve.

In [ ]:
def f(x, y):
    return x**2 + 3 * y**2

def grad_f(x, y):
    return np.array([2 * x, 6 * y])

grid = np.linspace(-3, 3, 400)
X, Y = np.meshgrid(grid, grid)
Z = f(X, Y)

arrow_grid = np.linspace(-3, 3, 12)
Xa, Ya = np.meshgrid(arrow_grid, arrow_grid)
Gx, Gy = 2 * Xa, 6 * Ya

fig, ax = plt.subplots(figsize=(7, 6))
ax.contour(X, Y, Z, levels=15, cmap='viridis')
ax.quiver(Xa, Ya, Gx, Gy, color='tab:red', alpha=0.6)
ax.set_aspect('equal'); ax.set_title('Level sets (blue) and gradient field (red).\nGradient is always perpendicular to the level curve.')
plt.show()

## 3. Hand-rolled gradient descent

Minimize $f(x, y) = x^2 + 3y^2$ from $(2.5, 2.5)$ with constant step size.

In [ ]:
path = [np.array([2.5, 2.5])]
lr = 0.1
for _ in range(40):
    x_cur = path[-1]
    path.append(x_cur - lr * grad_f(*x_cur))
path = np.array(path)

fig, ax = plt.subplots(figsize=(7, 6))
ax.contour(X, Y, Z, levels=20, cmap='viridis', alpha=0.6)
ax.plot(path[:, 0], path[:, 1], 'o-', color='tab:red', ms=4)
ax.set_aspect('equal'); ax.set_title('Gradient descent trajectory on f(x,y) = x^2 + 3y^2')
plt.show()

losses = [f(*p) for p in path]
plt.figure(figsize=(7, 3))
plt.semilogy(losses, marker='o', ms=3)
plt.xlabel('iteration'); plt.ylabel('loss (log)'); plt.title('Loss decrease (note zig-zag along the steep axis)')
plt.grid(True, which='both', alpha=0.3)
plt.show()

Notice that the trajectory zig-zags along the $y$-axis (the steep direction) — a classic symptom of ill-conditioning. Newton-style or preconditioned methods (Adam, RMSProp) compensate by scaling per-coordinate.

## 4. Numerical gradient check

The standard sanity check for any hand-derived (or autodiffed) gradient: compare against a finite-difference estimate.

In [ ]:
def g(x):
    # f(x) = 0.5 ||A x - b||^2; grad = A^T (A x - b)
    return 0.5 * np.sum((A_np @ x - b)**2)

def grad_g_analytic(x):
    return A_np.T @ (A_np @ x - b)

def grad_g_numeric(x, h=1e-5):
    n = len(x)
    g_num = np.zeros(n)
    for i in range(n):
        e = np.zeros(n); e[i] = h
        g_num[i] = (g(x + e) - g(x - e)) / (2 * h)   # central differences
    return g_num

A_np = rng.standard_normal((6, 4))
b = rng.standard_normal(6)
x_test = rng.standard_normal(4)

ga = grad_g_analytic(x_test)
gn = grad_g_numeric(x_test)
rel_err = np.linalg.norm(ga - gn) / max(np.linalg.norm(ga), 1e-12)
print(f'analytic gradient = {ga}')
print(f'numeric  gradient = {gn}')
print(f'relative error    = {rel_err:.2e}   (should be ~1e-8 with central differences)')